# Chapter 22: MLOps: CI/CD for AI Systems
**Part VI — MLOps & Production AI**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

MLflow experiment tracking, model monitoring, drift detection, and automated retraining.

## Learning Objectives
See Chapter 22 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Generate a data drift report


In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset

# Generate a data drift report
report = Report(metrics=[DataDriftPreset(), ClassificationPreset()])
report.run(reference_data=train_df, current_data=production_df,
           column_mapping=column_mapping)
report.save_html("drift_report.html")

# Check if significant drift was detected
result = report.as_dict()
drift_detected = result["metrics"][0]["result"]["dataset_drift"]
if drift_detected:
    print("ALERT: Dataset drift detected u2014 consider retraining")

## Fit-predict-score pattern


In [ ]:
# Fit-predict-score pattern
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
score = model.score(X_test, y_test)

## Data drift detection using Population Stability Index (PSI)


In [ ]:
# Data drift detection using Population Stability Index (PSI)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def compute_psi(expected, actual, buckets=10, eps=1e-10):
    """
    Compute Population Stability Index (PSI).
    PSI < 0.1  : no significant shift
    PSI 0.1-0.2: moderate shift, monitor
    PSI > 0.2  : significant shift, consider retraining
    """
    # Create buckets based on expected distribution
    breakpoints = np.quantile(expected, np.linspace(0, 1, buckets + 1))
    breakpoints[0] = -np.inf; breakpoints[-1] = np.inf
    
    expected_pcts = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pcts   = np.histogram(actual,   bins=breakpoints)[0] / len(actual)
    
    expected_pcts = np.clip(expected_pcts, eps, None)
    actual_pcts   = np.clip(actual_pcts,   eps, None)
    
    psi_vals = (actual_pcts - expected_pcts) * np.log(actual_pcts / expected_pcts)
    return psi_vals.sum(), psi_vals, breakpoints

np.random.seed(42)

# Simulate monthly feature distributions
training_data = np.random.normal(50, 10, 5000)         # baseline
month_no_drift  = np.random.normal(51, 10, 1000)       # slight noise
month_moderate  = np.random.normal(58, 12, 1000)       # moderate shift
month_severe    = np.random.normal(70, 15, 1000)       # severe shift

scenarios = [
    ("Month 1 (stable)", month_no_drift),
    ("Month 3 (moderate drift)", month_moderate),
    ("Month 6 (severe drift)", month_severe),
]

print("PSI Drift Detection Report")
print("=" * 50)
print(f"{'Month':<30} {'PSI':>8} {'Status':>15}")
print("-" * 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, data) in zip(axes, scenarios):
    psi, _, _ = compute_psi(training_data, data)
    if psi < 0.1:   status = "✅ Stable"
    elif psi < 0.2: status = "⚠️  Monitor"
    else:           status = "🔴 Retrain!"
    print(f"{name:<30} {psi:>8.4f} {status:>15}")
    
    ax.hist(training_data, bins=30, alpha=0.6, label='Training', color='#2E75B6', density=True)
    ax.hist(data, bins=30, alpha=0.6, label='Production', color='#C0392B', density=True)
    ax.set_title(f'{name}\nPSI = {psi:.4f}', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

print()
print("Thresholds: PSI < 0.1 = stable | 0.1-0.2 = monitor | > 0.2 = retrain")
plt.suptitle('Data Drift Detection via Population Stability Index', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ch22_drift.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 📚 Review Questions

See Chapter 22 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 22 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*